# ☕ Case #000: Who drank the last coffee? (step by step, in every framework)

The warm-up case from the session. It is small on purpose: three tools, one answer. You build the **same agent**
one step at a time, and you see what tools change.

| Step | What you do | What you see |
|---|---|---|
| 1 | **Create the agent**: a model plus instructions | Nothing runs yet |
| 2 | **Run it with no tools** | It guesses, or says it cannot know |
| 3 | **Create the tools** and give them to the agent | You call a tool by hand first |
| 4 | **Run it again** | 🔧 tool calls, then an answer with evidence |

Section 1 goes slowly and explains every step. Sections 2 to 4 repeat the same four steps in another framework,
so you can compare how each one writes them. Section 5 serves the tools over MCP.

| Section | Framework | Think of it as | Great for |
|---|---|---|---|
| 1 | **OpenAI Agents SDK** | A smart worker | Fast start, tools, handoffs, guardrails |
| 2 | **CrewAI** | A team with job titles | Role-based teams and task lists |
| 3 | **LangGraph** | A flowchart | Exact order, save points, human steps |
| 4 | **Google ADK** | Google's agent kit | Gemini (or any model), dev UI, agent-to-agent |
| 5 | **MCP** (not a framework) | USB-C for tools | Build a tool server once, plug it into any framework |

Every section uses the **same model** (from `agency_kit/kit.py`) and the **same three tool functions**, so you compare
the frameworks, not the tools. Run **Setup** first. After that, pick any section and run its steps **in order**.

**Before you start:** in the `AgencyCup` folder run `uv sync`, put `NVIDIA_API_KEY=nvapi-…` in `.env`, then open
this notebook in VS Code or Cursor and pick the `.venv` kernel. Run a cell with **Shift + Enter**.

In [ ]:
# ▶️ Setup: run me first
import sys
from agency_kit import kit
from agency_kit.practice_cases import COFFEE_BRIEF, coffee_log, desk_cam, who_owns  # the three tools, ready-made

INSTRUCTIONS = ("You are a detective. Collect evidence before you answer. "
                "Name who drank the last coffee and list your evidence.")
# Section 5 (MCP): how to start the practice tool server. Each framework runs it as a subprocess (stdio).
SERVER = {"command": sys.executable, "args": ["-m", "agency_kit.mcp_practice", "coffee"]}
kit.describe()
print(COFFEE_BRIEF)

---
## 1. OpenAI Agents SDK, step by step
An `Agent` is a model plus instructions plus tools. `Runner.run` runs the think → act → observe loop.

<small>🧠 `Agent(model=…)` · ⚡ `Runner.run()` · ✋ `function_tool` · 👥 handoffs, `.as_tool()` · 🛡️ guardrails, `needs_approval`</small>

### Step 1 · Create the agent
An agent needs a **name**, **instructions** in plain English, and a **model**. There are **no tools yet**.
Nothing runs in this cell: you only describe the agent.

In [ ]:
from agents import Agent, Runner, function_tool

rookie = Agent(name="Detective", instructions=INSTRUCTIONS, model=kit.openai_sdk_model())
print("Agent:", rookie.name, "· tools:", len(rookie.tools))
print("Instructions:", rookie.instructions)

### Step 2 · Run it with no tools
`Runner.run` sends the case to the model and waits for the answer. Read it closely: is there any **real evidence** in it?

In [ ]:
guess = await Runner.run(rookie, COFFEE_BRIEF)
print("🤷 Without tools:\n", guess.final_output)

> Without tools, an AI can only **guess** or **ask**. The case names nobody, so any name here is made up.
> A confident made-up answer is called a **hallucination**.

### Step 3 · Create the tools
A tool is a **normal Python function**. The model never sees the code inside. It reads only three things:
the **name**, the **inputs** (the type hints) and the **description** (the text in `"""quotes"""`).

Run the cell, then look at the last two lines: you can call a tool by hand, like any function.

In [ ]:
def coffee_log() -> str:
    """When the coffee machine last made a cup. Input: none."""
    return "Last cup made at 09:10. The pot is now empty."


def desk_cam(time: str) -> str:
    """What the camera at the coffee machine saw at one moment. Input: a time like "09:10"."""
    if time.strip() == "09:10":
        return "One person at the machine, holding a 'World's Best Dev' mug."
    return "Nobody at the machine."


def who_owns(item: str) -> str:
    """Who an object belongs to. Input: an item, like "red mug"."""
    owners = {"world's best dev mug": "Alex, the workshop facilitator", "red mug": "Sam the intern"}
    key = item.strip().lower().replace("’", "'").strip("'\" ")
    for name, owner in owners.items():
        if name in key or key in name:
            return f"That item belongs to {owner}."
    return "No owner found."


print("By hand:", coffee_log())
print("By hand:", desk_cam("09:10"))

**Now give the tools to the agent.** It is the same agent as step 1, plus one line: `tools=[...]`.
- `function_tool` turns a plain function into something the model can call.
- `kit.traced` prints a 🔧 line each time a tool runs, so you can watch the agent work.

In [ ]:
tools = [kit.traced(f) for f in (coffee_log, desk_cam, who_owns)]
detective = Agent(name="Detective", instructions=INSTRUCTIONS, model=kit.openai_sdk_model(),
                  tools=[function_tool(f) for f in tools])
print("Agent:", detective.name, "· tools:", [t.name for t in detective.tools])

### Step 4 · Run it again
The same question and the same instructions. Watch the 🔧 lines: the model **thinks**, **acts** (calls a tool),
**observes** what came back, and repeats until it can answer.

In [ ]:
result = await Runner.run(detective, COFFEE_BRIEF, max_turns=10)
print("\n🕵️ With tools:\n", result.final_output)

### Step 5 · What changed?
Scroll up and read the two answers side by side (steps 2 and 4). Then run this cell to count the tool calls and see what the agent did, in order.

In [ ]:
def tool_calls(run):
    return sum(type(item).__name__ == "ToolCallItem" for item in run.new_items)


print(f"🤷 Without tools: {tool_calls(guess)} tool calls. Nothing in that answer was checked.")
print(f"🕵️ With tools:    {tool_calls(result)} tool calls. Every fact in that answer came from a tool.")
print("\n📜 What the agent did, in order:")
for item in result.new_items:
    kind = type(item).__name__
    if kind == "ToolCallItem":
        print("  🔧 asked for", item.raw_item.name, item.raw_item.arguments)
    elif kind == "ToolCallOutputItem":
        print("     ↳ got back:", item.output)
    elif kind == "MessageOutputItem":
        print("  💬 answered")

The agent, the instructions and the model are the same in both runs. **Only the tools changed.**
Tools turn a guess into an answer you can check.

- 🗣️ **Talk:** which tool call gave the agent the name?
- ✏️ **Try:** in step 3, cut `Input: a time like "09:10".` from `desk_cam`'s description. Run steps 3 and 4 again. Does the agent still send a good time?

---
## 2. CrewAI: the same four steps
CrewAI thinks in **roles**: an agent has a role, a goal and a backstory, and does a `Task`.

<small>🧠 `Agent(role, goal, backstory)` · ⚡ `Crew(...).kickoff_async()` · ✋ `crewai.tools.tool` · 👥 `Task`s run in order</small>

### Step 1 · Create the agent
A role, a goal and a backstory. CrewAI also wants a `Task`: what to do, and what a good answer looks like. No tools yet.

In [ ]:
from crewai import Agent as CrewAgent, Crew, Task

rookie = CrewAgent(role="Detective", goal="Find out who drank the last coffee, with evidence",
                   backstory=INSTRUCTIONS, llm=kit.crewai_llm(), max_iter=10)
rookie_task = Task(description=COFFEE_BRIEF, expected_output="The person's name, and the evidence for it.", agent=rookie)

### Step 2 · Run it with no tools
A `Crew` runs the agents and their tasks. Here it is a crew of one.

In [ ]:
guess = await Crew(agents=[rookie], tasks=[rookie_task]).kickoff_async()
print("🤷 Without tools:\n", guess.raw)

### Step 3 · Give it the tools
CrewAI wants its own tool objects. `kit.crewai_tools` wraps the plain functions with `crewai.tools.tool`.

In [ ]:
tools = [kit.traced(f) for f in (coffee_log, desk_cam, who_owns)]
detective = CrewAgent(role="Detective", goal="Find out who drank the last coffee, with evidence",
                      backstory=INSTRUCTIONS, tools=kit.crewai_tools(tools), llm=kit.crewai_llm(), max_iter=10)
task = Task(description=COFFEE_BRIEF, expected_output="The person's name, and the evidence for it.", agent=detective)

### Step 4 · Run it again
Compare this answer with step 2: the role, the goal and the task are the same.

In [ ]:
result = await Crew(agents=[detective], tasks=[task]).kickoff_async()
print("\n🕵️ With tools:\n", result.raw)

---
## 3. LangGraph: the same four steps
LangGraph builds an agent as a **flowchart**. There is no ready-made agent: you **draw it yourself** with nodes (boxes) and edges (arrows).

<small>🧠 `ChatOpenAI(…).bind_tools()` · ⚡ `StateGraph(…).compile().ainvoke()` · ✋ plain functions in a `ToolNode` · 👥 nodes + edges · 🛡️ a human-check node</small>

### Step 1 · Create the agent
With no tools, the flowchart is one box: `think` calls the model, and then it ends.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode

plain_model = kit.langchain_model()  # the model, with no tools


async def answer(state: MessagesState):  # node: the model reads the case and answers
    return {"messages": [await plain_model.ainvoke([SystemMessage(INSTRUCTIONS), *state["messages"]])]}


graph = StateGraph(MessagesState)
graph.add_node("think", answer)
graph.add_edge(START, "think")
graph.add_edge("think", END)
rookie = graph.compile()
print(rookie.get_graph().draw_mermaid())  # the flowchart: start → think → end

### Step 2 · Run it with no tools

In [ ]:
guess = await rookie.ainvoke({"messages": [HumanMessage(COFFEE_BRIEF)]})
print("🤷 Without tools:\n", guess["messages"][-1].text)

### Step 3 · Give it the tools
Two changes. `bind_tools` tells the model which tools exist. An `act` box (`ToolNode`) runs the tools the model asks for.
The arrow from `act` back to `think` is the **agent loop**: look for it in the printed flowchart.

In [ ]:
tools = [kit.traced(f) for f in (coffee_log, desk_cam, who_owns)]
model = kit.langchain_model().bind_tools(tools)  # the model, told which tools it may call


async def think(state: MessagesState):  # node: the model reads everything so far and decides
    return {"messages": [await model.ainvoke([SystemMessage(INSTRUCTIONS), *state["messages"]])]}


def next_step(state: MessagesState):  # edge: did the model ask for a tool, or is it done?
    return "act" if state["messages"][-1].tool_calls else END


graph = StateGraph(MessagesState)
graph.add_node("think", think)
graph.add_node("act", ToolNode(tools))  # node: runs the tools the model asked for
graph.add_edge(START, "think")
graph.add_conditional_edges("think", next_step, ["act", END])
graph.add_edge("act", "think")  # observe the results, then think again
detective = graph.compile()
print(detective.get_graph().draw_mermaid())  # the flowchart grew a loop: think ⇄ act

### Step 4 · Run it again
Compare this answer with step 2. The straight line became a loop, and the guess became evidence.

In [ ]:
result = await detective.ainvoke({"messages": [HumanMessage(COFFEE_BRIEF)]}, {"recursion_limit": 25})
print("\n🕵️ With tools:\n", result["messages"][-1].text)

---
## 4. Google ADK: the same four steps
An `LlmAgent` looks a lot like the Agents SDK's `Agent`. `LiteLlm` (inside `kit.adk_model()`) lets ADK use an NVIDIA model instead of Gemini.

<small>🧠 `LlmAgent(model=…)` · ⚡ `InMemoryRunner(…).run_debug()` · ✋ plain functions · 👥 `sub_agents` · 🛡️ `before_tool_callback`</small>

### Step 1 · Create the agent
A name, a model and an instruction. No tools yet.

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner

rookie = LlmAgent(name="detective", model=kit.adk_model(), instruction=INSTRUCTIONS)

### Step 2 · Run it with no tools
ADK runs an agent through a **runner**. It gives back a list of events; the final answer is one of them.

In [ ]:
runner = InMemoryRunner(agent=rookie, app_name="coffee")  # keeps the session in memory
events = await runner.run_debug(COFFEE_BRIEF, quiet=True)  # every event: tool calls, tool results, answers
guess = [p.text for e in events if e.is_final_response() for p in e.content.parts if p.text and not p.thought]
print("🤷 Without tools:\n", "\n".join(guess))

### Step 3 · Give it the tools
ADK takes plain Python functions as they are. No wrapper is needed.

In [ ]:
tools = [kit.traced(f) for f in (coffee_log, desk_cam, who_owns)]
detective = LlmAgent(name="detective", model=kit.adk_model(), instruction=INSTRUCTIONS, tools=tools)

### Step 4 · Run it again
`kit.run_adk` is the runner lines from step 2, in one helper. Compare this answer with step 2.

In [ ]:
print("\n🕵️ With tools:\n", await kit.run_adk(detective, COFFEE_BRIEF))

---
## 5. MCP: one tool server, every framework
So far the tools were Python functions inside this notebook. With **MCP** they live in a separate **tool server**
(`agency_kit/mcp_practice.py`), and any framework can plug it in. The server runs the kit's copy of the three tools.

<small>🦵 Agents SDK `MCPServerStdio/StreamableHttp` · LangGraph `MultiServerMCPClient` tools in a `ToolNode` · ADK `McpToolset` · CrewAI `mcps=[…]`</small>

### Step 1 · Meet the server
Start the server, ask which tools it has, and call one by hand. There is no agent yet.

In [ ]:
from agents.mcp import MCPServerStdio

# The server can take 10+ s to start on Windows, so wait up to 60 s (the default is 5 s)
async with MCPServerStdio(params=SERVER, client_session_timeout_seconds=60) as server:
    for t in await server.list_tools():
        print(f"🔧 {t.name}: {t.description}")
    reply = await server.call_tool("desk_cam", {"time": "09:10"})
    print("\nBy hand, over MCP:", reply.content[0].text)

> The server answers with each tool's **name**, **description** and **inputs**. That is all the model needs, so you write no tool code.

### Step 2 · Plug the server into an agent
Pick your framework. Each cell is the agent from its section, with the tools now coming from the server.
(The calls happen inside the server process, so the 🔧 trace lines do not print here.)

In [ ]:
# OpenAI Agents SDK: give the SERVER to the agent
from agents import Agent, Runner
from agents.mcp import MCPServerStdio
# The server can take 10+ s to start on Windows, so wait up to 60 s (the default is 5 s)
async with MCPServerStdio(params=SERVER, client_session_timeout_seconds=60, cache_tools_list=True) as server:
    detective = Agent(name="Detective", instructions=INSTRUCTIONS, model=kit.openai_sdk_model(), mcp_servers=[server])
    result = await Runner.run(detective, COFFEE_BRIEF, max_turns=10)
print("🕵️ Agents SDK + MCP:", result.final_output)

In [ ]:
# LangGraph: ask the server for its tools, then build the same flowchart as section 3
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode

mcp_tools = await MultiServerMCPClient({"case": {**SERVER, "transport": "stdio"}}).get_tools()
mcp_model = kit.langchain_model().bind_tools(mcp_tools)


async def think_mcp(state: MessagesState):
    return {"messages": [await mcp_model.ainvoke([SystemMessage(INSTRUCTIONS), *state["messages"]])]}


graph = StateGraph(MessagesState)
graph.add_node("think", think_mcp)
graph.add_node("act", ToolNode(mcp_tools))
graph.add_edge(START, "think")
graph.add_conditional_edges("think", lambda s: "act" if s["messages"][-1].tool_calls else END, ["act", END])
graph.add_edge("act", "think")
result = await graph.compile().ainvoke({"messages": [HumanMessage(COFFEE_BRIEF)]}, {"recursion_limit": 25})
print("🕵️ LangGraph + MCP:", result["messages"][-1].text)

In [ ]:
# Google ADK: the toolset goes straight into the tool list
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from mcp import StdioServerParameters

toolset = McpToolset(connection_params=StdioConnectionParams(server_params=StdioServerParameters(**SERVER),
                                                             timeout=60))  # slow start on Windows (default 5 s)
detective = LlmAgent(name="detective", model=kit.adk_model(), instruction=INSTRUCTIONS, tools=[toolset])
print("🕵️ ADK + MCP:", await kit.run_adk(detective, COFFEE_BRIEF))
await toolset.close()

In [ ]:
# CrewAI: the server goes in mcps=[...]
from crewai import Agent as CrewAgent, Crew, Task
from crewai.mcp import MCPServerStdio as CrewMCPStdio

detective = CrewAgent(role="Detective", goal="Find out who drank the last coffee, with evidence", backstory=INSTRUCTIONS,
                      mcps=[CrewMCPStdio(**SERVER)], llm=kit.crewai_llm(), max_iter=10)
result = await Crew(agents=[detective], tasks=[Task(description=COFFEE_BRIEF, agent=detective,
                                              expected_output="The person's name, and the evidence for it.")]).kickoff_async()
print("🕵️ CrewAI + MCP:", result.raw)

---
## 🏁 Compare
- In every framework, what was the **one change** between the guess and the answer with evidence?
- Which framework needed the **least code**? Which one would you pick for a **team of roles**, or a **strict flow**?
- Look at the 🔧 traces: did every framework call the tools in the same order?